# TextAsset Deep Dive — Titles, Overlays & Lower-Thirds
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/editor/feature/text_asset_titles_overlays.ipynb)

---

Text makes or breaks visual storytelling. A well-styled title grabs attention. A clean lower-third adds context. A poorly positioned overlay distracts and looks unprofessional.

In this notebook, we'll explore every property of TextAsset in the VideoDB Editor—from basic text rendering to sophisticated multi-property compositions. You'll learn how to control fonts, colors, positioning, backgrounds, borders, shadows, and animations to create production-ready text overlays.

**What you'll learn:**

- How TextAsset works and its complete property set (`text`, `font`, `border`, `shadow`, `background`, `alignment`, `line_spacing`, `tabsize`)
- Font styling with the `Font` class (size, color, opacity, family)
- Visual depth using `Border` and `Shadow` classes
- Background boxes with the `Background` class and text alignment controls
- Precise positioning with the `Alignment` class (9 grid positions) and `Offset` fine-tuning
- Clip-level effects: transitions, opacity, and scale for animated text
- Real-world templates: lower-thirds, watermarks, subtitles, and title cards

---

## 📦 Step 1: Installing VideoDB Editor SDK

First, we'll install the VideoDB SDK.

In [ ]:
!pip -q install videodb

---

## 📦 Step 2: Establish Connection to VideoDB

Let's connect to VideoDB using our API key.

In [ ]:
import videodb
import os
from getpass import getpass
from videodb import play_stream

api_key = getpass("Please enter your VideoDB API Key: ")
os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()
coll = conn.get_collection()

print("✅ Connected to VideoDB successfully!")

Please enter your VideoDB API Key: ··········
✅ Connected to VideoDB successfully!


---

## 📦 Step 3: Upload Background Video

We'll upload one video to use as a background throughout this notebook. We'll mute and apply greyscale to keep focus on the text styling.

**Note:** Replace the URL with any cityscape or scenic video.

In [ ]:
# Upload background video
video = coll.upload(url="https://www.youtube.com/watch?v=RB9nyUyNI2s")
print(f"✅ Video uploaded: {video.id}")

# If you already have the video, use:
# video = coll.get_video("video_id")

✅ Video uploaded: m-z-019b49b9-8618-7510-9cc7-6e0133c88bf1


---

## 📦 Step 4: Import Editor Components

Now we'll import all Editor components needed for working with TextAsset. Pay attention to the TextAsset helper classes: `Font`, `Border`, `Shadow`, `Background`, `Alignment`, and their enums.

In [ ]:
from videodb.editor import (
    Timeline, Track, Clip, VideoAsset,
    TextAsset, Font, Border, Shadow, Background,
    Alignment, HorizontalAlignment, VerticalAlignment, TextAlignment,
    Filter, Position, Offset, Transition
)

print("✅ Editor components imported!")

✅ Editor components imported!


---

## 📦 Step 5: The `text` Property — Basic String Content

The `text` parameter is the core content. Let's start with simple text over a black background to see the most basic TextAsset.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#000000"
timeline.resolution = "1280x720"

# Minimal TextAsset — just text and font
text_asset = TextAsset(
    text="Hello World",
    font=Font(family="Clear Sans", size=48, color="#FFFFFF")
)

text_clip = Clip(asset=text_asset, duration=5)

track = Track()
track.add_clip(0, text_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/ac69c9ba-58ab-4388-9db5-e39c57f24fb4.m3u8


**What we see:** White text on black background, centered by default.

The `text` parameter accepts any string. Next, let's explore multiline text.

---

## 📦 Step 6: Multiline Text with `\n` and `line_spacing`

Use `\n` to create line breaks. The `line_spacing` parameter controls vertical spacing between lines (default: 1.0).

Let's compare different line_spacing values.

### line_spacing=0.8 (Tight)

In [ ]:
timeline = Timeline(conn)
timeline.background = "#000000"
timeline.resolution = "1280x720"

text_asset = TextAsset(
    text="""Line One \n Line Two \n Line Three""",
    font=Font(family="Clear Sans", size=48, color="#FFFFFF"),
    line_spacing=0.8  # Tighter spacing
)

text_clip = Clip(asset=text_asset, duration=5)
track = Track()
track.add_clip(0, text_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Tight spacing: {stream_url}")
play_stream(stream_url)

Tight spacing: https://play.videodb.io/v1/aa365cfa-8da6-43f3-9f3b-593a2c531336.m3u8


### line_spacing=1.5 (Spacious)

In [ ]:
timeline = Timeline(conn)
timeline.background = "#000000"
timeline.resolution = "1280x720"

text_asset = TextAsset(
    text="Line One \n Line Two \n Line Three",
    font=Font(family="Clear Sans", size=48, color="#FFFFFF"),
    line_spacing=1.5  # More space between lines
)

text_clip = Clip(asset=text_asset, duration=5)
track = Track()
track.add_clip(0, text_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Spacious: {stream_url}")
play_stream(stream_url)

Spacious: https://play.videodb.io/v1/405d4ba7-07d6-4726-9481-26e379756b5d.m3u8


**Key insight:** `line_spacing` is a multiplier. Values < 1.0 compress lines, values > 1.0 expand them. Use this for titles with subtitles or multi-line labels.

---

## 📦 Step 7: The `Font` Class — Typography Control

The `Font` class controls all text appearance. It has four main parameters:
- `family`: Font name (string)
- `size`: Font size in points (int/float)
- `color`: HTML hex color (string)
- `opacity`: Transparency 0.0-1.0 (float)

Let's explore each systematically.

### Font `size` Variations

Font size dramatically affects readability and emphasis. Let's compare small, medium, and large sizes.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"
timeline.resolution = "1280x720"

# Small text (24pt)
small_text = TextAsset(
    text="Small Text (24pt)",
    font=Font(family="Clear Sans", size=24, color="#FFFFFF"),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.top
    )
)

# Medium text (48pt)
medium_text = TextAsset(
    text="Medium Text (48pt)",
    font=Font(family="Clear Sans", size=48, color="#FFFFFF"),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.center
    )
)

# Large text (84pt)
large_text = TextAsset(
    text="Large Text (84pt)",
    font=Font(family="Clear Sans", size=84, color="#FFFFFF"),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.bottom
    )
)

small_clip = Clip(asset=small_text, duration=5, offset=Offset(x=0, y=0.1))
medium_clip = Clip(asset=medium_text, duration=5)
large_clip = Clip(asset=large_text, duration=5, offset=Offset(x=0, y=-0.1))

track = Track()
track.add_clip(0, small_clip)
track.add_clip(0, medium_clip)
track.add_clip(0, large_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Size comparison: {stream_url}")
play_stream(stream_url)

Size comparison: https://play.videodb.io/v1/bdc4705f-f9b9-4e14-98a9-79314f09eab0.m3u8


**Best practices:**
- **Small (18-30pt)**: Subtitles, fine print, secondary info
- **Medium (36-60pt)**: Body text, names, standard labels
- **Large (72-120pt)**: Titles, headlines, primary emphasis

### Font `color` Variations

Colors use HTML hex format (`#RRGGBB`). Let's see different color choices over our gray background.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"
timeline.resolution = "1280x720"

# Red text
red_text = TextAsset(
    text="Red Text",
    font=Font(family="Clear Sans", size=54, color="#FF0000"),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.top
    )
)

# Yellow text
yellow_text = TextAsset(
    text="Yellow Text",
    font=Font(family="Clear Sans", size=54, color="#FFFF00"),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.center
    )
)

# Cyan text
cyan_text = TextAsset(
    text="Cyan Text",
    font=Font(family="Clear Sans", size=54, color="#00FFFF"),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.bottom
    )
)

red_clip = Clip(asset=red_text, duration=5, offset=Offset(x=0, y=0.1))
yellow_clip = Clip(asset=yellow_text, duration=5)
cyan_clip = Clip(asset=cyan_text, duration=5, offset=Offset(x=0, y=-0.1))

text_track = Track()
text_track.add_clip(0, red_clip)
text_track.add_clip(0, yellow_clip)
text_track.add_clip(0, cyan_clip)
timeline.add_track(text_track)

stream_url = timeline.generate_stream()
print(f"Color comparison: {stream_url}")
play_stream(stream_url)

Color comparison: https://play.videodb.io/v1/47c60ddb-a479-471b-85dc-3a3e94774269.m3u8


**Key insight:** Color choice matters for readability. Over complex video backgrounds, high-contrast colors (white, yellow, cyan) work best. We'll add borders later to improve this further.

### Font `opacity` Variations

Opacity controls text transparency (0.0 = invisible, 1.0 = fully opaque). Useful for watermarks or subtle overlays.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"
timeline.resolution = "1280x720"


# Full opacity (1.0)
opaque_text = TextAsset(
    text="Opacity 1.0",
    font=Font(family="Clear Sans", size=60, color="#000000", opacity=1.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.top
    )
)

# Medium opacity (0.7)
medium_text = TextAsset(
    text="Opacity 0.7",
    font=Font(family="Clear Sans", size=60, color="#000000", opacity=0.7),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.center
    )
)

# Low opacity (0.5)
faint_text = TextAsset(
    text="Opacity 0.5",
    font=Font(family="Clear Sans", size=60, color="#000000", opacity=0.5),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.bottom
    )
)

opaque_clip = Clip(asset=opaque_text, duration=5, offset=Offset(x=0, y=0.1))
medium_clip = Clip(asset=medium_text, duration=5)
faint_clip = Clip(asset=faint_text, duration=5, offset=Offset(x=0, y=-0.1))

text_track = Track()
text_track.add_clip(0, opaque_clip)
text_track.add_clip(0, medium_clip)
text_track.add_clip(0, faint_clip)
timeline.add_track(text_track)

stream_url = timeline.generate_stream()
print(f"Opacity comparison: {stream_url}")
play_stream(stream_url)

Opacity comparison: https://play.videodb.io/v1/c41ee1c4-bea6-4775-bde3-d87dab456da1.m3u8


**Use cases:**
- **opacity=1.0**: Standard text, maximum readability
- **opacity=0.5-0.7**: Watermarks, subtle timestamps
- **opacity=0.2-0.4**: Background decorative text

---

## 📦 Step 8: The `Border` Class — Text Outlines

The `Border` class adds an outline around text characters, dramatically improving readability over complex backgrounds.

It has two parameters:
- `color`: HTML hex color
- `width`: Border thickness in pixels (float)

Let's explore border width variations.

### Border Width Comparison

In [ ]:
timeline = Timeline(conn)
timeline.background = "#FFFFFF"
timeline.resolution = "1280x720"

# Background Video
video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=5
)

# Thin border (1px)
thin_border = TextAsset(
    text="Border: 1px",
    font=Font(family="Clear Sans", size=54, color="#FFFF00"),
    border=Border(color="#000000", width=1.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.top
    )
)

# Medium border (3px)
medium_border = TextAsset(
    text="Border: 3px",
    font=Font(family="Clear Sans", size=54, color="#FFFF00"),
    border=Border(color="#000000", width=3.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.center
    )
)

# Thick border (6px)
thick_border = TextAsset(
    text="Border: 6px",
    font=Font(family="Clear Sans", size=54, color="#FFFF00"),
    border=Border(color="#000000", width=6.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.bottom
    )
)

thin_clip = Clip(asset=thin_border, duration=5, offset=Offset(x=0, y=0.1))
medium_clip = Clip(asset=medium_border, duration=5)
thick_clip = Clip(asset=thick_border, duration=5, offset=Offset(x=0, y=-0.1))

video_track = Track()
video_track.add_clip(0, video_clip)
timeline.add_track(video_track)

text_track = Track()
text_track.add_clip(0, thin_clip)
text_track.add_clip(0, medium_clip)
text_track.add_clip(0, thick_clip)
timeline.add_track(text_track)

stream_url = timeline.generate_stream()
print(f"Border width comparison: {stream_url}")
play_stream(stream_url)

Border width comparison: https://play.videodb.io/v1/56fe55bb-c4e8-475e-8af5-aac044259754.m3u8


**Best practices:**
- **1-2px**: Subtle definition, for large text
- **3-4px**: Standard readability, most common
- **5-8px**: Strong emphasis, stylized look

### Border Color Contrast

Border color creates contrast. Let's see white text with different border colors.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"
timeline.resolution = "1280x720"

video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=5
)

# Black border (classic)
black_border = TextAsset(
    text="Black Border",
    font=Font(family="Clear Sans", size=54, color="#FFFFFF"),
    border=Border(color="#000000", width=3.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.top
    )
)

# Blue border (accent)
blue_border = TextAsset(
    text="Blue Border",
    font=Font(family="Clear Sans", size=54, color="#FFFFFF"),
    border=Border(color="#0066FF", width=3.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.center
    )
)

# Red border (emphasis)
red_border = TextAsset(
    text="Red Border",
    font=Font(family="Clear Sans", size=54, color="#FFFFFF"),
    border=Border(color="#FF0000", width=3.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.bottom
    )
)

black_clip = Clip(asset=black_border, duration=5, offset=Offset(x=0, y=0.1))
blue_clip = Clip(asset=blue_border, duration=5)
red_clip = Clip(asset=red_border, duration=5, offset=Offset(x=0, y=-0.1))

video_track = Track()
video_track.add_clip(0, video_clip)
timeline.add_track(video_track)

text_track = Track()
text_track.add_clip(0, black_clip)
text_track.add_clip(0, blue_clip)
text_track.add_clip(0, red_clip)
timeline.add_track(text_track)

stream_url = timeline.generate_stream()
print(f"Border color comparison: {stream_url}")
play_stream(stream_url)

Border color comparison: https://play.videodb.io/v1/eb2a7605-6c75-46e7-9e07-31092b1a5f62.m3u8


**Key insight:** Black borders on light text provide maximum readability. Colored borders can match brand colors or create stylistic effects.

---

## 📦 Step 9: The `Shadow` Class — Adding Depth

The `Shadow` class creates a drop shadow behind text, adding depth and improving separation from the background.

It has three parameters:
- `color`: HTML hex color
- `x`: Horizontal offset in pixels (float)
- `y`: Vertical offset in pixels (float)

Shadows work by rendering a duplicate of the text offset from the original.

### Shadow Offset Variations

Let's explore different x/y offset combinations to see how shadow positioning affects depth.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#F0F0F0"
timeline.resolution = "1280x720"

# Small shadow (2px offset)
small_shadow = TextAsset(
    text="Small Shadow (2,2)",
    font=Font(family="Clear Sans", size=54, color="#FFFFFF"),
    border=Border(color="#000000", width=2.0),
    shadow=Shadow(color="#000000", x=2.0, y=2.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.top
    )
)

# Medium shadow (6px offset)
medium_shadow = TextAsset(
    text="Medium Shadow (6,6)",
    font=Font(family="Clear Sans", size=54, color="#FFFFFF"),
    border=Border(color="#000000", width=2.0),
    shadow=Shadow(color="#000000", x=6.0, y=6.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.center
    )
)

# Large shadow (12px offset)
large_shadow = TextAsset(
    text="Large Shadow (12,12)",
    font=Font(family="Clear Sans", size=54, color="#FFFFFF"),
    border=Border(color="#000000", width=2.0),
    shadow=Shadow(color="#000000", x=12.0, y=12.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.bottom
    )
)

small_clip = Clip(asset=small_shadow, duration=5, offset=Offset(x=0, y=0.1))
medium_clip = Clip(asset=medium_shadow, duration=5)
large_clip = Clip(asset=large_shadow, duration=5, offset=Offset(x=0, y=-0.1))

text_track = Track()
text_track.add_clip(0, small_clip)
text_track.add_clip(0, medium_clip)
text_track.add_clip(0, large_clip)
timeline.add_track(text_track)

stream_url = timeline.generate_stream()
print(f"Shadow offset comparison: {stream_url}")
play_stream(stream_url)

Shadow offset comparison: https://play.videodb.io/v1/95012bbe-474d-4528-a9e8-9b0f194c4f70.m3u8


**Key insight:** Larger offsets create more dramatic 3D effects. The shadow color should be darker than the background for maximum impact.

### Combining Border and Shadow

Border + Shadow together create professional, broadcast-quality text.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"
timeline.resolution = "1280x720"

video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=5
)

# Professional text: Border + Shadow
professional_text = TextAsset(
    text="Professional Text",
    font=Font(family="Clear Sans", size=72, color="#FFFFFF"),
    border=Border(color="#000000", width=3.0),
    shadow=Shadow(color="#333333", x=6.0, y=6.0),
    alignment=Alignment(
      horizontal=HorizontalAlignment.center,
      vertical=VerticalAlignment.top
    )
)

text_clip = Clip(asset=professional_text, duration=5, offset=Offset(x=0, y=0.1),)

video_track = Track()
video_track.add_clip(0, video_clip)
timeline.add_track(video_track)

text_track = Track()
text_track.add_clip(0, text_clip)
timeline.add_track(text_track)

stream_url = timeline.generate_stream()
print(f"Border + Shadow: {stream_url}")
play_stream(stream_url)

Border + Shadow: https://play.videodb.io/v1/9c8b4fb5-a44e-43e0-8d97-d9fc0dae6984.m3u8


---

## 📦 Step 10: The `Background` Class — Lower-Thirds and Boxes

The `Background` class creates a solid rectangular box behind text. This is essential for lower-thirds, name tags, and ensuring readability.

It has six parameters:
- `width`: Box width in pixels (int)
- `height`: Box height in pixels (int)
- `color`: HTML hex color (string)
- `opacity`: Box transparency 0.0-1.0 (float)
- `border_width`: Border around the box (float)
- `text_alignment`: How text aligns within the box (TextAlignment enum)

### Basic Background Box

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"
timeline.resolution = "1280x720"

video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=5
)

# Text with background box
text_with_bg = TextAsset(
    text="Background",
    font=Font(family="Clear Sans", size=48, color="#FFFFFF"),
    border=Border(color="#000000", width=2.0),
    background=Background(
        width=450,
        height=100,
        color="#1E3A8A",  # Deep blue
        opacity=0.9,
        border_width=3.0,
        text_alignment=TextAlignment.center
    ),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.top
    )
)

text_clip = Clip(asset=text_with_bg, duration=5, offset=Offset(x=0, y=0.1))

video_track = Track()
video_track.add_clip(0, video_clip)
timeline.add_track(video_track)

text_track = Track()
text_track.add_clip(0, text_clip)
timeline.add_track(text_track)

stream_url = timeline.generate_stream()
print(f"Background box: {stream_url}")
play_stream(stream_url)

Background box: https://play.videodb.io/v1/0754a309-507d-4f4f-825c-3094275f2dcb.m3u8


### text_alignment Within Background

The `text_alignment` parameter controls how text is positioned within the background box: `left`, `center`, or `right`.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"
timeline.resolution = "1280x720"

video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=5
)

# Left-aligned text in box
left_text = TextAsset(
    text="Left Aligned",
    font=Font(family="Clear Sans", size=40, color="#FFFFFF"),
    background=Background(
        width=500,
        height=80,
        color="#333333",
        opacity=0.9,
        text_alignment=TextAlignment.left
    ),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.top
    )
)

# Center-aligned text in box
center_text = TextAsset(
    text="Center Aligned",
    font=Font(family="Clear Sans", size=40, color="#FFFFFF"),
    background=Background(
        width=500,
        height=80,
        color="#333333",
        opacity=0.9,
        text_alignment=TextAlignment.center
    ),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.center
    )
)

# Right-aligned text in box
right_text = TextAsset(
    text="Right Aligned",
    font=Font(family="Clear Sans", size=40, color="#FFFFFF"),
    background=Background(
        width=500,
        height=80,
        color="#333333",
        opacity=0.9,
        text_alignment=TextAlignment.right
    ),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.bottom
    )
)

left_clip = Clip(asset=left_text, duration=5, offset=Offset(x=0, y=0.1))
center_clip = Clip(asset=center_text, duration=5)
right_clip = Clip(asset=right_text, duration=5, offset=Offset(x=0, y=-0.1))

video_track = Track()
video_track.add_clip(0, video_clip)
timeline.add_track(video_track)

text_track = Track()
text_track.add_clip(0, left_clip)
text_track.add_clip(0, center_clip)
text_track.add_clip(0, right_clip)
timeline.add_track(text_track)

stream_url = timeline.generate_stream()
print(f"Text alignment comparison: {stream_url}")
play_stream(stream_url)

Text alignment comparison: https://play.videodb.io/v1/6332b4c2-295c-45d7-86dc-db3e3c43736a.m3u8


---

## 📦 Step 11: The `Alignment` Class — Screen Positioning

The `Alignment` class positions text on screen using a 3×3 grid. It combines:
- `HorizontalAlignment`: left, center, right
- `VerticalAlignment`: top, center, bottom

This gives us 9 possible positions. Let's visualize the complete grid.

### The Complete Alignment Grid

We'll place text in all 9 positions to show the complete positioning system.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"
timeline.resolution = "1280x720"

# Create text for all 9 positions
positions = [
    (HorizontalAlignment.left, VerticalAlignment.top, "Top-Left"),
    (HorizontalAlignment.center, VerticalAlignment.top, "Top-Center"),
    (HorizontalAlignment.right, VerticalAlignment.top, "Top-Right"),
    (HorizontalAlignment.left, VerticalAlignment.center, "Mid-Left"),
    (HorizontalAlignment.center, VerticalAlignment.center, "Center"),
    (HorizontalAlignment.right, VerticalAlignment.center, "Mid-Right"),
    (HorizontalAlignment.left, VerticalAlignment.bottom, "Bot-Left"),
    (HorizontalAlignment.center, VerticalAlignment.bottom, "Bot-Center"),
    (HorizontalAlignment.right, VerticalAlignment.bottom, "Bot-Right"),
]

track = Track()

for h_align, v_align, label in positions:
    text_asset = TextAsset(
        text=label,
        font=Font(family="Clear Sans", size=60, color="#FFD700"),
        border=Border(color="#000000", width=2.0),
        alignment=Alignment(horizontal=h_align, vertical=v_align)
    )
    clip = Clip(asset=text_asset, duration=5)
    track.add_clip(0, clip)

timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Complete alignment grid: {stream_url}")
play_stream(stream_url)

Complete alignment grid: https://play.videodb.io/v1/dc2f95e7-0bd8-4093-8fe0-07860f7ce936.m3u8


---

## 📦 Step 12: Fine-Tuning with `Offset`

`Offset` lets us nudge text from its aligned position using x/y values:
- Range: -1.0 to 1.0 (relative to screen dimensions)
- `x`: Horizontal shift (0.1 = 10% right, -0.1 = 10% left)
- `y`: Vertical shift (0.1 = 10% down, -0.1 = 10% up)

Note: `Offset` is applied at the **Clip level** using `Position` and `Offset`, not in the TextAsset itself.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"
timeline.resolution = "1280x720"

# Base position: top-right
text_asset = TextAsset(
    text="Offset 0,0",
    font=Font(family="Clear Sans", size=48, color="#FFFFFF"),
    border=Border(color="#000000", width=2.0)
)

# No offset
base_clip = Clip(
    asset=text_asset,
    duration=5,
    position=Position.center
)

# Offset left and down
text_offset = TextAsset(
    text="Offset -0.1, 0.1",
    font=Font(family="Clear Sans", size=48, color="#FFD700"),
    border=Border(color="#000000", width=2.0)
)

offset_clip = Clip(
    asset=text_offset,
    duration=5,
    position=Position.center,
    offset=Offset(x=-0.1, y=0.1)  # 10% left, 10% down
)

track = Track()
track.add_clip(0, base_clip)
track.add_clip(0, offset_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Offset demonstration: {stream_url}")
play_stream(stream_url)

Offset demonstration: https://play.videodb.io/v1/5e7273fe-7bf3-4058-87fc-bf6a3f9b31ad.m3u8


**📝 Insight:** Offset gives you pixel-perfect control for tweaking positions. Useful when a corner position is close but you need a small nudge (e.g., moving text away from screen edges).

### Bottom-Left with Offset Adjustment

In [ ]:
timeline = Timeline(conn)

video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=5
)

# Create text at bottom-left with offset to move it inward
text_asset = TextAsset(
    text="Sample Text",
    font=Font(family="Clear Sans", size=36, color="#FFFFFF"),
    background=Background(width=350, height=100, color="#000000", opacity=0.8),
    alignment=Alignment(
        horizontal=HorizontalAlignment.left,
        vertical=VerticalAlignment.bottom
    )
)

text_clip = Clip(
    asset=text_asset,
    duration=5,
    position=Position.bottom_left,
    offset=Offset(x=0.05, y=-0.05)  # 5% right, 5% up from corner
)

video_track = Track()
video_track.add_clip(0, video_clip)
timeline.add_track(video_track)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, text_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Bottom-left with offset: {stream_url}")
play_stream(stream_url)

Bottom-left with offset: https://play.videodb.io/v1/5fe9ef6c-a94a-4eae-be21-d24ca7f9d331.m3u8


---

## Step 13: Advanced Properties - `tabsize`

The **`tabsize`** parameter (default 4) controls tab character width when your text contains `\t`. Useful for aligned columns or structured data displays.

### Tabsize Comparison (4 vs 8)

In [ ]:
timeline = Timeline(conn)
timeline.background = "#000000"

# Tabsize 4 (default)
text_tab4 = TextAsset(
    text="City:\tTokyo",
    font=Font(family="Clear Sans", size=32, color="#FFFFFF"),
    background=Background(width=200, height=100, color="#1E1E1E", opacity=0.9),
    tabsize=4,
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.top
    )
)

clip_tab4 = Clip(
    asset=text_tab4,
    duration=5,
    offset = Offset(x=0,y=0.1)
)

# Tabsize 8
text_tab8 = TextAsset(
    text="City:\tTokyo",
    font=Font(family="Clear Sans", size=32, color="#FFFFFF"),
    background=Background(width=200, height=100, color="#1E1E1E", opacity=0.9),
    tabsize=8,
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.bottom
    )
)

clip_tab8 = Clip(
    asset=text_tab8,
    duration=5,
    offset = Offset(x=0,y=-0.1)
)

track = Track()
track.add_clip(0, clip_tab4)
track.add_clip(0, clip_tab8)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Tabsize comparison: {stream_url}")
play_stream(stream_url)

Tabsize comparison: https://play.videodb.io/v1/99fd0182-e701-40b9-bd11-3ef8fc94af71.m3u8


**📝 Insight:** Tabsize is subtle but important for data displays. Larger tabsize creates wider spacing between columns. Test with your actual text to find the right alignment.

---

## Step 14: Clip-Level Effects on Text

TextAsset properties control the text itself. But **Clip-level** properties like `transition`, `opacity`, and `scale` add animation and effects to the entire text clip.

### Fade Transition

In [ ]:
timeline = Timeline(conn)

video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=7
)

# Text with fade in and fade out
text_asset = TextAsset(
    text="TOKYO",
    font=Font(family="Clear Sans", size=72, color="#FFD700", opacity=1.0)
)

text_clip = Clip(
    asset=text_asset,
    duration=5,
    transition= Transition(
        in_="fade" ,
        out="fade" ,
        duration=2 )
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(1, text_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Text with fade transition: {stream_url}")
play_stream(stream_url)

Text with fade transition: https://play.videodb.io/v1/d0b45f0c-ea5c-4aa1-aa32-ddafe51c1285.m3u8


### Opacity Animation

In [ ]:
timeline = Timeline(conn)
video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=8
)

# Full opacity text
text_full = TextAsset(
    text="Full Opacity (1.0)",
    font=Font(family="Clear Sans", size=48, color="#00FFFF"),
    border=Border(color="#000000", width=2.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.top
    )
)

clip_full = Clip(
    asset=text_full,
    duration=8,
    opacity=1.0,  # Clip-level opacity
    offset = Offset(x=0,y=0.1)
)

# Half opacity text
text_half = TextAsset(
    text="Half Opacity (0.5)",
    font=Font(family="Clear Sans", size=48, color="#00FFFF"),
    border=Border(color="#000000", width=2.0),    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.center
    )
)

clip_half = Clip(
    asset=text_half,
    duration=8,
    opacity=0.5  # Clip-level opacity
)

# Quarter opacity text
text_quarter = TextAsset(
    text="Quarter Opacity (0.25)",
    font=Font(family="Clear Sans", size=48, color="#00FFFF"),
    border=Border(color="#000000", width=2.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.center,
        vertical=VerticalAlignment.bottom
    )
)

clip_quarter = Clip(
    asset=text_quarter,
    duration=8,
    opacity=0.25,  # Clip-level opacity
    offset = Offset(x=0,y=-0.1)
)

video_track = Track()
video_track.add_clip(0, video_clip)
timeline.add_track(video_track)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, clip_full)
track.add_clip(0, clip_half)
track.add_clip(0, clip_quarter)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Clip-level opacity: {stream_url}")
play_stream(stream_url)

Clip-level opacity: https://play.videodb.io/v1/6a376949-0421-45c7-9e47-61b1545d4b35.m3u8


**📝 Note:** This is **Clip-level opacity** (affects the entire text element), different from **Font.opacity** (affects only the text characters).

### Scale Effect

In [ ]:
timeline = Timeline(conn)
video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=7
)

# Small scale (0.7x)
text_small = TextAsset(
    text="Small (0.7x)",
    font=Font(family="Clear Sans", size=48, color="#FF69B4"),
    border=Border(color="#000000", width=2.0),
)

clip_small = Clip(
    asset=text_small,
    duration=7,
    scale=0.7,  # 70% of original size
    offset = Offset(x=0,y=-0.3)
)

# Normal scale (1.0x)
text_normal = TextAsset(
    text="Normal (1.0x)",
    font=Font(family="Clear Sans", size=48, color="#FF69B4"),
    border=Border(color="#000000", width=2.0)
)

clip_normal = Clip(
    asset=text_normal,
    duration=7,
    scale=1.0  # 100% original size
)

# Large scale (1.5x)
text_large = TextAsset(
    text="Large (1.5x)",
    font=Font(family="Clear Sans", size=48, color="#FF69B4"),
    border=Border(color="#000000", width=2.0)
)

clip_large = Clip(
    asset=text_large,
    duration=7,
    scale=1.5,  # 150% of original size
    offset = Offset(x=0,y=0.3)
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, clip_small)
track.add_clip(0, clip_normal)
track.add_clip(0, clip_large)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Scale variations: {stream_url}")
play_stream(stream_url)

Scale variations: https://play.videodb.io/v1/f5b89704-221b-42da-b7c8-227afbe3c2b1.m3u8


**📝 Insight:** Scale is powerful for emphasis or de-emphasis. You can also combine it with animations (scale + fade) for dynamic title reveals.

---

## Step 15: Practical Templates & Recipes

Now that we've explored every property, let's build **real-world templates** you can reuse in your projects.

### Template 1: Branded Lower-Third

In [ ]:
timeline = Timeline(conn)
video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=5
)

# Professional lower-third with brand colors
lower_third = TextAsset(
    text="Dr. Sarah Chen: Tech Officer",
    font=Font(family="Clear Sans", size=36, color="#FFFFFF", opacity=1.0),
    background=Background(
        width=600,
        height=120,
        color="#1E88E5",  # Brand blue
        opacity=0.95
    ),
    border=Border(color="#FFC107", width=4.0),  # Gold accent
    shadow=Shadow(color="#000000", x=3, y=3),
    alignment=Alignment(
        horizontal=HorizontalAlignment.left,
        vertical=VerticalAlignment.bottom
    )
)

lower_third_clip = Clip(
    asset=lower_third,
    duration=6,
    offset=Offset(x=0.03, y=-0.05)
)

# Fade in and out
lower_third_clip.transition = Transition(
    in_="fade",
    out="fade",
    duration=2
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, lower_third_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Branded lower-third: {stream_url}")
play_stream(stream_url)

Branded lower-third: https://play.videodb.io/v1/25a56282-b960-4b1b-a6df-4af246792d75.m3u8


### Template 2: Watermark / Logo Text

In [ ]:
timeline = Timeline(conn)
video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=5
)

# Subtle watermark in corner
watermark = TextAsset(
    text="© MyBrand 2024",
    font=Font(family="Clear Sans", size=24, color="#FFFFFF", opacity=0.5),
    border=Border(color="#000000", width=1.0),
    alignment=Alignment(
        horizontal=HorizontalAlignment.right,
        vertical=VerticalAlignment.top
    )
)

watermark_clip = Clip(
    asset=watermark,
    duration=5,
    offset=Offset(x=-0.02, y=0.02),  # Inset from corner
    opacity=0.7  # Additional transparency
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, watermark_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Watermark: {stream_url}")
play_stream(stream_url)

Watermark: https://play.videodb.io/v1/182e0337-63fa-4a71-8b67-5d3e89979f86.m3u8


### Template 3: Centered Title Card

In [ ]:
timeline = Timeline(conn)
timeline.background = "#000000"  # Black background
timeline.resolution = "1280x720"

# Big, bold centered title
title_card = TextAsset(
    text="TOKYO // JAPAN",
    font=Font(family="Clear Sans", size=96, color="#FFFFFF", opacity=1.0),
    line_spacing=1.4,
    border=Border(color="#FFD700", width=5.0),
    shadow=Shadow(color="#000000", x=6, y=6)
)

title_clip = Clip(
    asset=title_card,
    duration=4
)

# Dramatic fade in/out
title_clip.transition = Transition(
    in_ = "fade",
    out = "fade",
    duration = 2
)

track = Track()
track.add_clip(0, title_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Title card: {stream_url}")
play_stream(stream_url)

Title card: https://play.videodb.io/v1/c3c451ae-5500-4255-a7a7-d262a2d62ddd.m3u8


### Template 4: Subtitle Style with Background

In [ ]:
timeline = Timeline(conn)
video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=6
)

# Subtitle with semi-transparent black background
subtitle = TextAsset(
    text="Welcome to Tokyo, the bustling capital of Japan.",
    font=Font(family="Clear Sans", size=32, color="#FFFFFF", opacity=1.0),
    background=Background(
        width=1100,
        height=80,
        color="#000000",
        opacity=0.75,
        text_alignment=TextAlignment.center
        ),
        alignment= Alignment(
            horizontal=HorizontalAlignment.center,
            vertical=VerticalAlignment.bottom
    )
)

subtitle_clip = Clip(
    asset=subtitle,
    duration=6,
    offset=Offset(x=0, y=-0.08)
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, subtitle_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Subtitle style: {stream_url}")
play_stream(stream_url)

Subtitle style: https://play.videodb.io/v1/ad532679-d80c-4160-bcb1-dfb8439e4c6e.m3u8


### Template 5: Timestamp Overlay

In [ ]:
timeline = Timeline(conn)
video_clip = Clip(
    asset=VideoAsset(id=video.id, start=0, volume=0),
    duration=5
)

# Timestamp in corner
timestamp = TextAsset(
    text="2024-03-15 14:30 JST",
    font=Font(family="Clear Sans", size=28, color="#FFFFFF", opacity=0.9),
    background=Background(
        width=300,
        height=70,
        color="#000000",
        opacity=0.6,
        text_alignment=TextAlignment.center
    ),
    alignment=Alignment(
        horizontal=HorizontalAlignment.left,
        vertical=VerticalAlignment.bottom
    )
)

timestamp_clip = Clip(
    asset=timestamp,
    duration=5,
    offset=Offset(x=0.02, y=-0.02)
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, timestamp_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Timestamp overlay: {stream_url}")
play_stream(stream_url)

Timestamp overlay: https://play.videodb.io/v1/06355b90-dff1-4acf-9855-fdc30de0d352.m3u8


---

## Step 16: Wrap-Up & Next Steps

🎉 **You've mastered TextAsset!**

### What We Covered

We explored **every TextAsset property** systematically:

1. **`text`** - String content with `\n` for multiline
2. **`line_spacing`** - Control spacing between lines (0.8, 1.5, 2.0)
3. **`Font`** class - `size`, `color`, `opacity`, `family`
4. **`Border`** class - `width`, `color` for outlines
5. **`Shadow`** class - `x`, `y` offsets, `color` for depth
6. **`Background`** class - `width`, `height`, `color`, `opacity`, `text_alignment`
7. **`Alignment`** class - `horizontal` (left/center/right), `vertical` (top/center/bottom)
8. **`Offset`** - Fine-tune position with x/y relative values
9. **`tabsize`** - Tab character width for structured text
10. **Clip-level effects** - `transition` (fade), `opacity`, `scale` for animation

### Key Insights

✅ **Font.opacity vs Clip.opacity**: Font affects text only; Clip affects the entire element (text + background + border)  
✅ **Position + Offset**: Start with Position enum for coarse placement, use Offset for fine-tuning  
✅ **Background.text_alignment**: Controls text alignment *within* the background box  
✅ **Border + Shadow combo**: Professional look with edge definition + depth  
✅ **Transition for polish**: Fade in/out makes text feel less abrupt

### Actionable Next Steps

🔹 **Try different font families** - The examples used "Clear Sans", but experiment with others  
🔹 **Combine properties creatively** - Mix scale animations with opacity changes  
🔹 **Build your own templates** - Create reusable recipes for your brand style  
🔹 **Test color schemes** - Brand colors, contrast ratios, accessibility  
🔹 **Explore timing** - Vary `duration`, `start` times for dynamic sequences

---

## 🚀 Ready to Build!

You now have complete knowledge of **TextAsset** for titles, overlays, and lower-thirds. Experiment with the properties, combine them creatively, and build stunning text overlays for your videos.

**Happy creating!** 🎬